# ⬡ Go Research Lab — Colab 서버

**실행 순서: 셀 1 → 2 → 3 → 4 순서대로 실행하세요.**

- 셀 3 실행 후 출력되는 `https://xxxx.ngrok-free.app` URL을 복사
- HTML의 서버 주소 입력창에 붙여넣으면 연결 완료

> ⚠️ Colab 무료 플랜: 세션 최대 12시간 / 유휴 시 자동 종료  
> 세션이 끊기면 이 노트북을 다시 처음부터 실행하세요.

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  셀 1 — GPU 확인 + KataGo 설치                          │
# └─────────────────────────────────────────────────────────┘
import subprocess, os

# GPU 확인
print('=== GPU 상태 ===')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo '❌ GPU 없음 (런타임 유형을 GPU로 변경하세요)'

# KataGo 설치 디렉토리
os.makedirs('/content/katago', exist_ok=True)

# KataGo CUDA 바이너리 다운로드 (GPU용)
if not os.path.exists('/content/katago/katago'):
    print('\n=== KataGo 다운로드 중... ===')
    !wget -q 'https://github.com/lightvector/KataGo/releases/download/v1.15.3/katago-v1.15.3-cuda12.1-linux-x86_64.zip' \
         -O /content/katago/katago.zip
    !unzip -q /content/katago/katago.zip -d /content/katago/
    !chmod +x /content/katago/katago
    print('✅ KataGo 설치 완료')
else:
    print('✅ KataGo 이미 설치됨')

# KataGo 모델 다운로드 (b18 네트워크 — 강력하고 범용적)
if not os.path.exists('/content/katago/model.bin.gz'):
    print('\n=== KataGo 모델 다운로드 중... (약 200MB, 시간이 걸립니다) ===')
    !wget -q 'https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b18c384nbt-s9996604416-d4316597426.bin.gz' \
         -O /content/katago/model.bin.gz
    print('✅ 모델 다운로드 완료')
else:
    print('✅ 모델 이미 존재함')

print('\n✅ 셀 1 완료 → 셀 2로 이동하세요')

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  셀 2 — KataGo config 파일 생성                          │
# └─────────────────────────────────────────────────────────┘

config_content = '''
# KataGo GTP Config (Colab GPU용)
logDir = /content/katago/logs
logAllGTPCommunication = false
logSearchInfo = false

# GPU 설정
cudaUseFP16 = true
cudaUseNHWC = true

# 탐색 설정
maxVisits = 100000
numSearchThreads = 4

# 룰 설정
rules = tromp-taylor
'''

import os
os.makedirs('/content/katago/logs', exist_ok=True)
with open('/content/katago/default_gtp.cfg', 'w') as f:
    f.write(config_content)

print('✅ config 파일 생성 완료')
print('✅ 셀 2 완료 → 셀 3으로 이동하세요')

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  셀 3 — GitHub에서 서버 코드 받아서 실행 + ngrok 터널    │
# └─────────────────────────────────────────────────────────┘

# ★ 여기에 본인 GitHub 정보를 입력하세요
GITHUB_USER = 'YOUR_GITHUB_USERNAME'   # 예: 'joon4606'
GITHUB_REPO = 'YOUR_REPO_NAME'         # 예: 'go-research-lab'
NGROK_TOKEN = 'YOUR_NGROK_AUTH_TOKEN'  # https://dashboard.ngrok.com 에서 발급

# ─────────────────────────────────────────────────────────
import subprocess, os, time, threading

# GitHub에서 서버 파일 다운로드
SERVER_URL = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/main/server_colab_09.py'
print(f'서버 코드 받는 중: {SERVER_URL}')
!wget -q '{SERVER_URL}' -O /content/server_colab_09.py

if not os.path.exists('/content/server_colab_09.py'):
    print('❌ 서버 파일 다운로드 실패. GitHub URL을 확인하세요.')
else:
    print('✅ 서버 파일 다운로드 완료')

# 필요 패키지 설치
!pip install -q flask flask-cors pyngrok

# ngrok 인증
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# Flask 서버를 별도 스레드에서 실행
def run_server():
    os.system('python /content/server_colab_09.py')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print('서버 시작 중... (5초 대기)')
time.sleep(5)

# ngrok 터널 생성
try:
    # 기존 터널 정리
    ngrok.kill()
except:
    pass

tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print('\n' + '='*60)
print(f'✅ 서버 공개 URL:')
print(f'   {public_url}')
print('='*60)
print('\n👆 이 URL을 복사해서 HTML 서버 주소 입력창에 붙여넣으세요.')
print('   (끝에 슬래시 없이 그대로 복사)')

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  셀 4 — 서버 상태 확인 (선택)                            │
# └─────────────────────────────────────────────────────────┘
import requests

try:
    # 간단한 ping 테스트
    res = requests.post(f'{public_url}/analyze',
                        json={'moves': [], 'lz_visits': 100},
                        timeout=15)
    data = res.json()
    print(f'✅ 서버 응답 정상')
    print(f'   최선수: {data.get("best_move")}  승률: {data.get("winrate")}%')
except Exception as e:
    print(f'❌ 서버 응답 실패: {e}')
    print('   셀 3을 다시 실행해보세요.')

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  셀 5 — 세션 유지 (Colab 자동 종료 방지, 선택)           │
# └─────────────────────────────────────────────────────────┘
# Colab은 일정 시간 동안 활동이 없으면 세션을 종료합니다.
# 아래 셀을 실행해두면 30분마다 ping을 보내 세션을 유지합니다.
# (무료 플랜 최대 12시간 제한은 여전히 적용됩니다)

import time, threading

def keep_alive():
    while True:
        time.sleep(1800)  # 30분마다
        try:
            import requests
            requests.post(f'{public_url}/analyze',
                          json={'moves': [], 'lz_visits': 50}, timeout=10)
            print(f'[keep-alive] ping sent')
        except:
            pass

ka_thread = threading.Thread(target=keep_alive, daemon=True)
ka_thread.start()
print('✅ 세션 유지 활성화 (30분마다 ping)')